# Forecast Using the Chronos Family

## Import libraries

In [ ]:
import pandas as pd

from foundationforecast import FoundationForecast


## Load the dataset 

The DataFrame must include at least the following columns:
- unique_id: Unique identifier for each time series (string)
- ds: Date column (datetime format)
- y: Target variable for forecasting (float format)

The pandas frequency will be inferred from the ds column, if not provided.
If the seasonality is not provided, it will be inferred based on the frequency. 
If the horizon is not set, it will default to 2 times the inferred seasonality.

In [ ]:
df = pd.read_csv(
    "https://timecopilot.s3.amazonaws.com/public/data/events_pageviews.csv",
    parse_dates=["ds"],
)
df.head()


## Plot the data

In [ ]:
FoundationForecast.plot(df)

## Import the Chronos class 

In [ ]:
from foundationforecast.models.timegpt import TimeGPT



## Create a FoundationForecast


In [ ]:
timegpt_av_models = [
    "timegpt-1",
    "timegpt-1-long-horizon",
    "timegpt-2-mini",
    "timegpt-2",
    "timegpt-2-pro",
]
timegpt_models = [TimeGPT(model=model, alias=model) for model in timegpt_av_models]

ff = FoundationForecast(models=models)


## Generate forecast 

You can optionally specify the following parameters:
- freq: The frequency of your data (e.g., 'D' for daily, 'M' for monthly)
- h: The forecast horizon, which is the number of periods to predict
- seasonality: The seasonal period of your data, which can be inferred if not provided


In [ ]:
level = [20, 40, 60, 80]
cv_df = ff.cross_validation(df=df, h=12, level=level) 

In [ ]:
ff.plot(df, cv_df.drop(columns=["cutoff", "y"]), level=[80])

In [ ]:
cv_df.head()

## Evaluation

In [ ]:
from functools import partial

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mase, scaled_crps

In [ ]:
eval_df = evaluate(
    cv_df.drop(columns=["cutoff"]), 
    train_df=df.query("ds <= '2024-08-31'"), 
    metrics=[partial(mase, seasonality=12), scaled_crps],
    level=level,
)
eval_df.groupby("metric").mean(numeric_only=True).T.sort_values(by="scaled_crps").round(3)